# Car Price Prediction — Milestone 2: Data Preparation

**Dataset:** [Car Price Prediction Challenge — Kaggle](https://www.kaggle.com/datasets/deepcontractor/car-price-prediction-challenge)

Upload `car_price_dataset.csv` to Colab via: *Files → Upload to session storage*

---
**Pipeline order (data-leakage safe):**
1. Load & inspect raw data  
2. Data cleaning & imputation  
3. Outlier investigation  
4. **Train / Test split** ← done before any engineering  
5. Feature engineering (hand-crafted features)  
6. Analysis of new features  
7. Encoding categorical variables  
8. Scaling via sklearn Pipeline  
9. Feature selection  
10. Export

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import permutation_importance

# All random operations in this notebook use this seed for full reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams.update({'figure.dpi': 120})
PAL = sns.color_palette('Set2')

print('Setup complete. RANDOM_SEED =', RANDOM_SEED)

## 2. Load & Inspect Raw Data

In [ ]:
df_raw = pd.read_csv('car_price_dataset.csv')
print('Shape:', df_raw.shape)
df_raw.head(10)

In [ ]:
print('Column dtypes:')
print(df_raw.dtypes)
print('\nUnique values per column:')
print(df_raw.nunique().sort_values())

## 3. Data Cleaning

### 3.1 Hidden Missing Values

This dataset uses `"-"` as a placeholder for missing values rather than `NaN`.
The `Levy` (vehicle import tax) column is most affected (~30% of rows).

In [ ]:
dash_pct = (df_raw == '-').mean() * 100
dash_pct = dash_pct[dash_pct > 0].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 3.2))
bars = ax.bar(dash_pct.index, dash_pct.values, color=PAL[0], width=0.5)
ax.bar_label(bars, fmt='%.1f%%', padding=4, fontsize=10)
ax.set_ylabel('Rows with placeholder (%)')
ax.set_title('Hidden Missing Values per Column (dash used instead of NaN)')
ax.set_ylim(0, dash_pct.max() * 1.45)
sns.despine(); plt.tight_layout(); plt.show()
print(dash_pct)

### 3.2 Type Conversion & Imputation

In [ ]:
df = df_raw.copy()

# Replace all dash placeholders with proper NaN
df.replace('-', np.nan, inplace=True)

# Mileage: "186005 km" → 186005
df['Mileage'] = pd.to_numeric(
    df['Mileage'].astype(str).str.replace(' km', '', regex=False)
                             .str.replace(',', '', regex=False), errors='coerce')

# Engine volume: "2.0 Turbo" → 2.0  (Turbo flag extracted separately later)
df['Engine volume'] = pd.to_numeric(
    df['Engine volume'].astype(str).str.extract(r'(\d+\.?\d*)')[0], errors='coerce')

# Levy: remove thousands comma, cast to float
df['Levy'] = pd.to_numeric(
    df['Levy'].astype(str).str.replace(',', '', regex=False), errors='coerce')

# Price: cast to numeric — rows where it fails are dropped (target cannot be imputed)
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
n_before = len(df)
df.dropna(subset=['Price'], inplace=True)
print(f'Rows dropped — unparseable Price: {n_before - len(df)}')

# Levy: impute with median (preferred over mean — Levy is right-skewed)
levy_med = df['Levy'].median()
df['Levy'].fillna(levy_med, inplace=True)
print(f'Levy: imputed {(df_raw["Levy"] == "-").sum()} missing values with median = {levy_med:.0f}')

# Remaining numeric NaNs: impute with column median
for col in df.select_dtypes(include=np.number).columns:
    n = df[col].isnull().sum()
    if n > 0:
        df[col].fillna(df[col].median(), inplace=True)
        print(f'  {col}: {n} values imputed')

# Categorical NaNs: drop rows (fabricating a manufacturer or fuel type is invalid)
n_before = len(df)
df.dropna(subset=df.select_dtypes('object').columns, inplace=True)
print(f'Rows dropped — missing categorical: {n_before - len(df)}')
print(f'\nClean dataset shape: {df.shape}')
print(f'Total remaining NaNs: {df.isnull().sum().sum()}')

## 4. Outlier Investigation

Both ends of the price range are examined before applying any filter.

In [ ]:
df_before_filter = df.copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(df['Price'].clip(upper=150000), bins=80, color='#5B8DB8', edgecolor='none', alpha=0.9)
axes[0].set_title('Price — Linear Scale')
axes[0].set_xlabel('Price ($)'); axes[0].set_ylabel('Count')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].hist(np.log1p(df['Price']), bins=70, color=PAL[2], edgecolor='none', alpha=0.9)
axes[1].set_title('log(1 + Price) — full range visible')
axes[1].set_xlabel('log(1 + Price)'); axes[1].set_ylabel('Count')
plt.suptitle('Price Distribution Before Outlier Filtering', fontweight='bold')
sns.despine(); plt.tight_layout(); plt.show()

print('Lowest 10 prices:',  df['Price'].nsmallest(10).values)
print('Highest 10 prices:', df['Price'].nlargest(10).values)

In [ ]:
# LOWER bound only: prices < $500 are clear data entry errors (e.g. $1, $2)
# UPPER bound: NOT applied — luxury/exotic cars at $100k+ are valid listings.
# Removing them would introduce systematic underestimation bias for the premium segment.
n_before = len(df)
df = df[df['Price'] >= 500].copy()
print(f'Removed {n_before - len(df)} rows with Price < $500')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(np.log1p(df_before_filter['Price']), bins=60, color=PAL[3], edgecolor='none', alpha=0.9)
axes[0].set_title(f'Before (n={len(df_before_filter):,})'); axes[0].set_xlabel('log(1+Price)'); axes[0].set_ylabel('Count')
axes[1].hist(np.log1p(df['Price']), bins=60, color=PAL[4], edgecolor='none', alpha=0.9)
axes[1].set_title(f'After removing Price < $500 (n={len(df):,})'); axes[1].set_xlabel('log(1+Price)'); axes[1].set_ylabel('Count')
plt.suptitle('Effect of Lower-Bound Outlier Filtering', fontweight='bold')
sns.despine(); plt.tight_layout(); plt.show()

## 5. Preliminary Analysis — Feature Distributions & Correlations

In [ ]:
num_feats = ['Prod. year', 'Mileage', 'Engine volume', 'Levy', 'Airbags', 'Doors']
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, feat, c in zip(axes.flatten(), num_feats, PAL):
    ax.hist(df[feat].clip(upper=df[feat].quantile(0.99)), bins=35,
            color=c, edgecolor='none', alpha=0.9)
    ax.set_title(feat); ax.set_xlabel(feat); ax.set_ylabel('Count')
plt.suptitle('Distributions of Key Numeric Features', fontweight='bold')
sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
num_df = df.select_dtypes(include=np.number)
num_df = num_df[[c for c in num_df.columns if c != 'Price'] + ['Price']]
corr   = num_df.corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, linewidths=0.4, ax=ax,
            cbar_kws={'label': 'Pearson r', 'shrink': 0.75}, annot_kws={'size': 8})
ax.set_title('Correlation Matrix of Original Numeric Features', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

print('Correlations with Price (sorted):')
print(corr['Price'].drop('Price').sort_values().to_string())

## 6. Train / Test Split

**This split happens before any feature engineering** — fitting transformers on the full
dataset would allow test-set statistics to influence training (data leakage).

An **80/20 split stratified by price decile** ensures both partitions contain
a proportional share of low-, mid-, and high-value vehicles.

In [ ]:
df['_bin'] = pd.qcut(df['Price'], q=10, labels=False, duplicates='drop')
X = df.drop(columns=['Price', '_bin'])
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=df['_bin'])

print(f'Training samples : {len(X_train):,}  ({len(X_train)/len(X)*100:.0f}%)')
print(f'Test samples     : {len(X_test):,}   ({len(X_test)/len(X)*100:.0f}%)')

fig, ax = plt.subplots(figsize=(9, 4.5))
bins = np.linspace(np.log1p(y.min()), np.log1p(y.max()), 50)
ax.hist(np.log1p(y_train), bins=bins, alpha=0.65,
        label=f'Train  n={len(y_train):,}', color=PAL[0])
ax.hist(np.log1p(y_test),  bins=bins, alpha=0.65,
        label=f'Test   n={len(y_test):,}',  color=PAL[3])
ax.set_xlabel('log(1 + Price)'); ax.set_ylabel('Count')
ax.set_title('Target Distribution: Train vs Test — Stratified 80/20 Split', fontweight='bold')
ax.legend()
sns.despine(); plt.tight_layout(); plt.show()

print('\nTrain — price statistics:'); print(y_train.describe().round(0))
print('\nTest  — price statistics:'); print(y_test.describe().round(0))

## 7. Feature Engineering

Five hand-crafted features are derived from existing columns.
All are computed **after** the split and applied identically to train and test.

| Feature | Formula | Domain reasoning |
|---|---|---|
| `Car_Age` | 2024 − Prod. year | Depreciation is time-driven; age outperforms raw year |
| `Km_Per_Year` | Mileage ÷ (Car_Age + 1) | Normalises wear intensity by ownership duration |
| `Has_Turbo` | 1 if original engine string contains "Turbo" | Turbocharged engines command a measurable price premium |
| `Airbag_Tier` | Binned count: 0→0, 1-4→1, 5-8→2, 9+→3 | Safety tier reduces the effect of outlier airbag configurations |
| `Value_Score` | (Engine_vol × 1000 − Mileage × 0.01) ÷ (Car_Age + 1) | Composite metric: captures engine size relative to age and usage |

In [ ]:
# Store the raw engine string before cleaning — needed to extract the Turbo flag
raw_engine_strings = df_raw['Engine volume'].astype(str)

def add_features(X_in, raw_eng_series):
    """Add domain-informed features. Accepts raw engine series explicitly to avoid
    hidden global dependencies and ensure the function is self-contained."""
    o = X_in.copy()
    o['Car_Age']     = (2024 - o['Prod. year'].astype(int)).clip(lower=0)
    o['Km_Per_Year'] = o['Mileage'] / (o['Car_Age'] + 1)
    o['Has_Turbo']   = (raw_eng_series.loc[X_in.index]
                        .str.lower().str.contains('turbo').astype(int))
    o['Airbag_Tier'] = pd.cut(o['Airbags'], bins=[-1, 0, 4, 8, 100],
                               labels=[0, 1, 2, 3]).astype(int)
    o['Value_Score'] = (o['Engine volume'] * 1000 - o['Mileage'] * 0.01
                        ) / (o['Car_Age'] + 1)
    return o

X_train = add_features(X_train, raw_engine_strings)
X_test  = add_features(X_test,  raw_engine_strings)

NEW_FEATS = ['Car_Age', 'Km_Per_Year', 'Has_Turbo', 'Airbag_Tier', 'Value_Score']
print('New features created:', NEW_FEATS)
print()
print('Descriptive statistics (training set):')
print(X_train[NEW_FEATS].describe().round(2))

## 8. Analysis of New Features

Each new feature is analysed using the same methods applied in Milestone 1:
distribution histograms, relationship to the target variable, and correlation analysis.

In [ ]:
# Distributions
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for i, (ax, feat) in enumerate(zip(axes.flatten(), NEW_FEATS)):
    ax.hist(X_train[feat].clip(upper=X_train[feat].quantile(0.98)),
            bins=32, color=PAL[i % 6], edgecolor='none', alpha=0.9)
    ax.set_title(feat); ax.set_xlabel(feat); ax.set_ylabel('Count')
axes.flatten()[-1].set_visible(False)
plt.suptitle('Distributions of New Hand-Crafted Features (Training Set)', fontweight='bold')
sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
# Regression plots — new continuous features vs Price
sample = X_train.copy(); sample['Price'] = y_train.values
sample = sample.sample(min(2500, len(sample)), random_state=RANDOM_SEED)

reg_pairs = [('Car_Age','Car Age (years)'),('Km_Per_Year','Km Per Year'),('Value_Score','Value Score')]
colors_r  = ['#2196F3','#4CAF50','#9C27B0']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (feat, label), col in zip(axes, reg_pairs, colors_r):
    x  = sample[feat].values; yy = sample['Price'].values
    ok = np.isfinite(x) & np.isfinite(yy)
    sl, ic, r, _, _ = stats.linregress(x[ok], yy[ok])
    xl = np.linspace(x[ok].min(), x[ok].max(), 100); yl = sl*xl + ic
    ax.scatter(x, yy, alpha=0.14, s=8, color=col)
    ax.plot(xl, yl, color='red', lw=2, label=f'r = {r:.2f}')
    ax.fill_between(xl, yl - yy[ok].std()*0.35, yl + yy[ok].std()*0.35,
                    alpha=0.13, color='red')
    ax.set_xlabel(label); ax.set_ylabel('Price ($)'); ax.set_title(f'{label} vs Price')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
    ax.legend(fontsize=9)
plt.suptitle('New Continuous Features vs Price — Regression Plots', fontweight='bold')
sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
# Violin plots — categorical new features
tp = pd.DataFrame({
    'Has_Turbo':   X_train['Has_Turbo'].map({0: 'Non-Turbo', 1: 'Turbo'}),
    'Airbag_Tier': X_train['Airbag_Tier'],
    'Price':       y_train.values
})
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.violinplot(data=tp, x='Has_Turbo',   y='Price', palette='Set2',
               ax=axes[0], inner='quartile', cut=0)
axes[0].set_title('Price by Engine Type'); axes[0].set_xlabel('Engine Type'); axes[0].set_ylabel('Price ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
sns.violinplot(data=tp, x='Airbag_Tier', y='Price', palette='Set2',
               ax=axes[1], inner='quartile', cut=0)
axes[1].set_title('Price by Airbag Safety Tier  (0=none → 3=high)')
axes[1].set_xlabel('Airbag Tier'); axes[1].set_ylabel('Price ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
plt.suptitle('Categorical New Features vs Price — Violin Plots', fontweight='bold')
sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
# KDE — price density by car age group
tp2 = pd.DataFrame({'Car_Age': X_train['Car_Age'], 'Price': y_train.values})
tp2['Age Group'] = pd.qcut(tp2['Car_Age'], q=3,
                            labels=['Young (0–6 yr)','Mid (7–13 yr)','Old (14+ yr)'])
fig, ax = plt.subplots(figsize=(9, 4.5))
for group, color in zip(['Young (0–6 yr)','Mid (7–13 yr)','Old (14+ yr)'], PAL):
    sns.kdeplot(np.log1p(tp2[tp2['Age Group']==group]['Price']),
                ax=ax, label=group, fill=True, alpha=0.28, color=color)
ax.set_xlabel('log(1 + Price)'); ax.set_ylabel('Density')
ax.set_title('Price Density by Car Age Group — KDE on log-price', fontweight='bold')
ax.legend(title='Age Group')
sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
# Correlation heatmap — new features + Price
nc = pd.DataFrame({f: X_train[f] for f in NEW_FEATS})
nc['Price'] = y_train.values
fig, ax = plt.subplots(figsize=(7.5, 6.5))
sns.heatmap(nc.corr(), annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Pearson r', 'shrink': 0.85})
ax.set_title('Correlation Matrix: New Features + Price', fontweight='bold')
plt.tight_layout(); plt.show()

## 9. Encoding Categorical Variables

**Why not Ordinal Encoding for Manufacturer/Model?**  
Ordinal encoding assigns Toyota=0, BMW=1, Ford=2 — imposing a fake numeric order.
The model would incorrectly treat the manufacturer difference as a distance.

**Target Encoding (high-cardinality: Manufacturer, Model, Color)**  
Each category is replaced by its mean Price in the **training set only**.
BMW → ~$45,000 because that reflects real market value. No leakage.

**One-Hot Encoding (low-cardinality: Fuel type, Gear box type, etc.)**  
Creates binary indicator columns. No ordering assumed.

In [ ]:
cat_cols  = X_train.select_dtypes('object').columns.tolist()
card      = {c: X_train[c].nunique() for c in cat_cols}
high_card = [c for c, n in card.items() if n > 8]
low_card  = [c for c, n in card.items() if n <= 8]

print('Target Encoding  (high cardinality):')
for c in high_card: print(f'  {c:25s} {card[c]:3d} unique values')
print('\nOne-Hot Encoding (low cardinality):')
for c in low_card:  print(f'  {c:25s} {card[c]:3d} unique values')

# ── Target Encoding ───────────────────────────────────────────────────────────
global_mean = y_train.mean()
target_maps = {}
for col in high_card:
    target_maps[col] = y_train.groupby(X_train[col].astype(str)).mean().to_dict()
    X_train[col] = X_train[col].astype(str).map(target_maps[col]).fillna(global_mean)
    X_test[col]  = X_test[col].astype(str).map(target_maps[col]).fillna(global_mean)

# Proof: show mean price per manufacturer
mfr_means = pd.Series(target_maps['Manufacturer']).sort_values(ascending=False)
fig, ax   = plt.subplots(figsize=(10, 4))
ax.bar(mfr_means.index, mfr_means.values, color=PAL[0], edgecolor='white')
ax.set_xlabel('Manufacturer'); ax.set_ylabel('Mean Training Price ($)')
ax.set_title('Target Encoding — Mean Price per Manufacturer (training set only)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
plt.xticks(rotation=40, ha='right'); sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
# ── One-Hot Encoding ──────────────────────────────────────────────────────────
X_train = pd.get_dummies(X_train, columns=low_card, drop_first=True)
X_test  = pd.get_dummies(X_test,  columns=low_card, drop_first=True)
# Align: test may be missing a dummy column if a rare category only appears in train
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

# Convert bool dummies to int (sklearn requires numeric input)
bool_cols = X_train.select_dtypes(include='bool').columns.tolist()
X_train[bool_cols] = X_train[bool_cols].astype(int)
X_test[bool_cols]  = X_test[bool_cols].astype(int)

print(f'Feature count after encoding: {X_train.shape[1]}')
print(f'All columns numeric: {X_train.select_dtypes(exclude=np.number).shape[1] == 0}')

## 10. Scaling via sklearn Pipeline + ColumnTransformer

### Why a Pipeline?
Manual scaling is error-prone — easy to accidentally call `fit_transform` on the test set.
A Pipeline makes the correct order **structural**: impossible to leak test statistics into training.

### Why MinMaxScaler?
Maps all features to **[0, 1]** while preserving the exact shape of each distribution.
Chosen over StandardScaler because our features include heavily right-skewed columns
(Mileage, Levy, Km_Per_Year) where the standard deviation is not a meaningful scale unit.

### Why SimpleImputer inside the pipeline?
Target encoding maps unseen test categories to `NaN` as a safe fallback.
Adding an imputer inside the pipeline handles this cleanly and maintains the
single-fit / single-transform contract.

In [ ]:
all_feature_cols = X_train.columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('impute_scale', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),  # handles residual NaNs
            ('scaler',  MinMaxScaler())                     # normalises to [0, 1]
        ]), all_feature_cols)
    ],
    remainder='drop'  # explicit: no unhandled column
)

preprocessing_pipeline = Pipeline(steps=[('preprocessing', preprocessor)])

# Fit on training set ONLY, then transform both
X_train_arr = preprocessing_pipeline.fit_transform(X_train)
X_test_arr  = preprocessing_pipeline.transform(X_test)

X_train_sc = pd.DataFrame(X_train_arr, columns=all_feature_cols, index=X_train.index)
X_test_sc  = pd.DataFrame(X_test_arr,  columns=all_feature_cols, index=X_test.index)

print('Pipeline fitted successfully.')
print(f'X_train_sc : {X_train_sc.shape}')
print(f'X_test_sc  : {X_test_sc.shape}')
print(f'NaNs       : {X_train_sc.isnull().sum().sum()}  (must be 0)')

In [ ]:
# Visualise scaling effect on 3 key features
demo = ['Mileage', 'Car_Age', 'Levy']
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for i, feat in enumerate(demo):
    axes[0, i].hist(X_train[feat], bins=35, color=PAL[i], edgecolor='none', alpha=0.9)
    axes[0, i].set_title(f'{feat} — Original'); axes[0, i].set_xlabel(feat); axes[0, i].set_ylabel('Count')
    axes[1, i].hist(X_train_sc[feat], bins=35, color=PAL[i+3], edgecolor='none', alpha=0.9)
    axes[1, i].set_title(f'{feat} — Scaled [0, 1]')
    axes[1, i].set_xlabel(f'{feat} (normalised)'); axes[1, i].set_ylabel('Count')
plt.suptitle('MinMaxScaler via Pipeline — Shape Preserved, Range Normalised to [0, 1]',
             fontweight='bold')
sns.despine(); plt.tight_layout(); plt.show()

## 11. Feature Selection — Permutation Importance

A `GradientBoostingRegressor` is trained and **Permutation Importance** computed.
This measures how much the model's error increases when a feature's values are randomly shuffled.
A large increase = the model depended on that feature.  Near zero = noise.

**Threshold: importance > 0.002**  
A genuine cutoff that reduces dimensionality. A threshold of `> 0` is meaningless —
it would keep every feature regardless of quality.

In [ ]:
gbm = GradientBoostingRegressor(
    n_estimators=100, max_depth=4, learning_rate=0.1, random_state=RANDOM_SEED)
gbm.fit(X_train_sc, y_train)

perm = permutation_importance(
    gbm, X_train_sc, y_train,
    n_repeats=10, random_state=RANDOM_SEED, n_jobs=-1)

perm_s = pd.Series(perm.importances_mean,
                   index=X_train_sc.columns).sort_values(ascending=False)
print(f'Features before selection: {len(perm_s)}')
print('\nAll scores:')
print(perm_s.round(5).to_string())

In [ ]:
IMPORTANCE_THRESHOLD = 0.002
selected = perm_s[perm_s >  IMPORTANCE_THRESHOLD].index.tolist()
dropped  = perm_s[perm_s <= IMPORTANCE_THRESHOLD].index.tolist()

print(f'Threshold        : {IMPORTANCE_THRESHOLD}')
print(f'Features KEPT    : {len(selected)}')
print(f'Features DROPPED : {len(dropped)}')
print(f'\nSelected: {selected}')

X_train_final = X_train_sc[selected].copy()
X_test_final  = X_test_sc[selected].copy()
print(f'\nFinal shapes — Train: {X_train_final.shape}  |  Test: {X_test_final.shape}')

In [ ]:
# Bar chart with threshold line
fig, ax = plt.subplots(figsize=(10, max(5, len(perm_s) * 0.34)))
bar_c = ['#27AE60' if v > IMPORTANCE_THRESHOLD else '#BDC3C7'
          for v in perm_s.sort_values().values]
perm_s.sort_values().plot.barh(ax=ax, color=bar_c, edgecolor='white')
ax.axvline(IMPORTANCE_THRESHOLD, color='red', linestyle='--', lw=2,
           label=f'Threshold = {IMPORTANCE_THRESHOLD}  '
                 f'({len(selected)} kept  |  {len(dropped)} dropped)')
ax.set_xlabel('Mean Permutation Importance (MSE increase when shuffled)')
ax.set_title('Feature Selection — Green = Selected  |  Grey = Dropped', fontweight='bold')
ax.legend(); sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
# Final correlation heatmap
plot_feats = selected[:12] if len(selected) >= 12 else selected
cf = X_train_final[plot_feats].copy()
cf['Price'] = y_train.values
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(cf.corr(), dtype=bool))
sns.heatmap(cf.corr(), mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.4, ax=ax,
            cbar_kws={'label': 'Pearson r', 'shrink': 0.8}, annot_kws={'size': 8})
ax.set_title(f'Final Correlation Matrix — Top {len(plot_feats)} Selected Features + Price',
             fontsize=12, fontweight='bold', pad=10)
plt.tight_layout(); plt.show()

## 12. Export Prepared Data

In [ ]:
X_train_final.to_csv('X_train_prepared.csv', index=False)
X_test_final.to_csv('X_test_prepared.csv',  index=False)
y_train.to_csv('y_train_prepared.csv',       index=False)
y_test.to_csv('y_test_prepared.csv',         index=False)

print('Saved:')
print(f'  X_train_prepared.csv  {X_train_final.shape}')
print(f'  X_test_prepared.csv   {X_test_final.shape}')
print(f'  y_train_prepared.csv  {y_train.shape}')
print(f'  y_test_prepared.csv   {y_test.shape}')